# Quai Nord · essai d'un réseau de neurones séquentiel (GRU)

**La question :** un réseau qui lit *la suite* des passages de la ligne depuis le matin prédit-il mieux
les perturbations que la régression logistique ?

**La comparaison** (sur exactement les mêmes jours) :
1. référence : le taux de la ligne ;
2. logistique actuelle ;
3. logistique + état de la ligne (un chiffre : la part de perturbés depuis le matin) ;
4. **réseau séquentiel** : GRU sur les 20 derniers passages + fiche du train.

**Avant de lancer :** menu *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Tout exécuter*.
Cet essai ne touche ni au site ni à la machine virtuelle.

## 1. Préparation : le code du projet
On récupère le code de Quai Nord pour utiliser **exactement** les mêmes fonctions que le modèle de chaque nuit.

In [ ]:
import os, sys
if not os.path.exists("quai-nord"):
    !git clone -q https://github.com/Aly2001/quai-nord.git
sys.path.insert(0, "quai-nord")

import numpy as np
import pandas as pd
import tensorflow as tf
import entrainer_modele as em      # le modèle logistique de chaque nuit
import sequences as sq             # la fabrication des séquences (même code que sur la VM)
print("TensorFlow", tf.__version__, "· GPU :", tf.config.list_physical_devices("GPU") or "aucun (plus lent)")

## 2. Les données
Le fichier `retards_hdf.csv` est fabriqué chaque nuit par la VM. Deux façons de l'avoir ici :
- **A. depuis le bucket** (après l'avoir copié depuis la VM) : connexion à ton compte Google, puis copie ;
- **B. à la main** : bouton *Importer* dans le panneau Fichiers, à gauche.

In [ ]:
FICHIER = "retards_hdf.csv"
if not os.path.exists(FICHIER):
    from google.colab import auth
    auth.authenticate_user()
    !gsutil cp gs://quai-nord-transfert-aly/donnees/retards_hdf.csv .

On garde les mêmes passages que le modèle de chaque nuit : les TER des Hauts-de-France, **constatés**, avec une heure prévue.
Puis on fabrique les mêmes variables (heure, jour, tranche de position) et on écarte les journées incomplètes.

In [ ]:
colonnes = ["date_service", "trip_id", "ligne", "heure_prevue", "ordre_arret", "retard_minutes",
            "supprime", "perturbe", "pluie_mm", "rafales_kmh", "type_observation"]
brut = pd.read_csv(FICHIER, sep=";", usecols=colonnes, low_memory=False)
df = brut[(brut["type_observation"] == "CONSTATE") & brut["heure_prevue"].notna()
          & brut["trip_id"].str.contains(":TER:", na=False)].drop(columns="type_observation")
df = em.preparer_variables(df)
jours = em.jours_complets(df, aujourd_hui=df["date"].max() + pd.Timedelta(days=1))
df = df[df["date"].isin(jours)].reset_index(drop=True)
print(f"{len(df):,} passages · {len(jours)} journées complètes ({jours[0]:%d/%m} → {jours[-1]:%d/%m})")
print(f"Part de perturbés : {df['perturbe'].mean():.1%}")

## 3. Les entrées du réseau

**La séquence** : pour chaque passage, les 20 derniers passages *déjà jugés* de sa ligne ce jour-là,
connus 30 min avant le départ du train. Tenseur de forme **(N, 20, 6)** : 20 passages × 6 nombres
(présent, retard, perturbé, supprimé, il y a combien de temps, position). Le petit matin est complété par des zéros au début ;
la colonne « présent » dit au réseau quelles lignes sont de vrais passages.

On calcule aussi « l'état de la ligne » de la logistique, avec la même règle des 30 min.

In [ ]:
%time SEQ = sq.construire_sequences(df)
df = em.etat_de_la_ligne(df)          # garde l'ordre des lignes : SEQ[i] correspond toujours à df.iloc[i]
print("Séquences :", SEQ.shape, f"· {SEQ.nbytes / 1e6:.0f} Mo")
i = int(np.argmax(SEQ[:, :, 0].sum(axis=1)))
print("Exemple, train", df.loc[i, "trip_id"], "à", df.loc[i, "heure_prevue"], "(5 derniers passages connus) :")
pd.DataFrame(SEQ[i][-5:], columns=sq.INFOS).round(3)

**La fiche du train** : la ligne, l'heure et le jour deviennent des **numéros** (pour les embeddings) ;
la position, la pluie et les rafales sont **mises à l'échelle** avec la moyenne et l'écart-type
des jours d'apprentissage **seulement** (sinon : fuite d'information).

In [ ]:
def preparer_fiche(apprentissage, *autres):
    lignes_connues = apprentissage["ligne"].value_counts()
    vocab = {l: k + 1 for k, l in enumerate(lignes_connues[lignes_connues >= em.FREQUENCE_MIN].index)}  # 0 = ligne rare ou inconnue
    nombres = ["ordre_arret", "pluie_mm", "rafales_kmh"]
    mediane = apprentissage[nombres].median()
    moyenne = apprentissage[nombres].fillna(mediane).mean()
    ecart = apprentissage[nombres].fillna(mediane).std().replace(0, 1)

    def fiche(t):
        return {
            "ligne": t["ligne"].map(vocab).fillna(0).to_numpy("int32"),
            "heure": t["heure"].to_numpy("int32"),
            "jour": t["jour_semaine"].to_numpy("int32"),
            "nombres": ((t[nombres].fillna(mediane) - moyenne) / ecart).to_numpy("float32"),
        }
    return len(vocab) + 1, [fiche(t) for t in (apprentissage, *autres)]

## 4. Le réseau (la structure du schéma)

```
Séquence (20, 6) ─► GRU(32) ──────────────┐
Ligne ─► Embedding(8) ────────────────────┤
Heure ─► Embedding(4) ────────────────────┼─► Concaténer (50) ─► Dense(32, ReLU) ─► Dropout(0,2) ─► Dense(1, sigmoïde) ─► p
Jour  ─► Embedding(3) ────────────────────┤
Position, pluie, rafales (3) ─────────────┘
```

In [ ]:
from tensorflow.keras import layers, Model

def construire_reseau(nb_lignes):
    seq = layers.Input((sq.LONGUEUR, sq.NB_INFOS), name="sequence")
    ligne = layers.Input((), dtype="int32", name="ligne")
    heure = layers.Input((), dtype="int32", name="heure")
    jour = layers.Input((), dtype="int32", name="jour")
    nombres = layers.Input((3,), name="nombres")

    journee = layers.GRU(32, name="gru")(seq)                         # (B, 32) : le résumé de la journée
    e_ligne = layers.Embedding(nb_lignes, 8, name="emb_ligne")(ligne)  # (B, 8)
    e_heure = layers.Embedding(24, 4, name="emb_heure")(heure)         # (B, 4)
    e_jour = layers.Embedding(7, 3, name="emb_jour")(jour)             # (B, 3)

    x = layers.Concatenate(name="concat")([journee, e_ligne, e_heure, e_jour, nombres])   # (B, 50)
    x = layers.Dense(32, activation="relu", name="cachee")(x)          # (B, 32)
    x = layers.Dropout(0.2)(x)
    p = layers.Dense(1, activation="sigmoid", name="sortie")(x)        # (B, 1) : la probabilité

    modele = Model([seq, ligne, heure, jour, nombres], p)
    modele.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
    return modele

construire_reseau(nb_lignes=70).summary()

## 5. L'apprentissage
- erreur : **log-loss** (`binary_crossentropy`), la même que la régression logistique ;
- **Adam**, par paquets de 1 024 exemples ;
- **arrêt précoce** : le dernier jour d'apprentissage sert de validation ; dès que l'erreur sur ce jour
  ne baisse plus pendant 2 époques, on s'arrête et on garde les meilleurs poids.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

def entrainer_reseau(apprentissage, validation, test, graine):
    tf.keras.utils.set_random_seed(graine)                 # les poids de départ sont tirés au hasard
    nb_lignes, (f_app, f_val, f_test) = preparer_fiche(apprentissage, validation, test)
    entree = lambda t, f: {"sequence": SEQ[t.index.to_numpy()], **f}
    modele = construire_reseau(nb_lignes)
    modele.fit(entree(apprentissage, f_app), apprentissage["perturbe"].to_numpy("float32"),
               validation_data=(entree(validation, f_val), validation["perturbe"].to_numpy("float32")),
               epochs=15, batch_size=1024, verbose=0,
               callbacks=[EarlyStopping(patience=2, restore_best_weights=True)])
    return modele.predict(entree(test, f_test), batch_size=4096, verbose=0).ravel()

## 6. Une « nuit » rejouée
Comme la VM chaque nuit : apprendre sur les jours anciens, tester sur les **4 derniers jours jamais vus**,
puis calculer le score de Brier et le **gain** sur le taux de la ligne.

In [ ]:
def une_nuit(jours_nuit, graines):
    test = df[df["date"].isin(jours_nuit[-em.JOURS_TEST:])]
    apprentissage = df[df["date"].isin(jours_nuit[:-em.JOURS_TEST])]
    y = test["perturbe"].to_numpy(float)
    brier = lambda p: float(((np.asarray(p) - y) ** 2).mean())

    constante = apprentissage["perturbe"].mean()
    par_ligne = apprentissage.groupby("ligne")["perturbe"].agg(["sum", "count"])
    taux = (par_ligne["sum"] + 20 * constante) / (par_ligne["count"] + 20)
    reference = brier(test["ligne"].map(taux).fillna(constante))
    gain = lambda b: 100 * (1 - b / reference)

    logi = em.construire_modele().fit(em.preparer(apprentissage), apprentissage["perturbe"])
    p_logi = logi.predict_proba(em.preparer(test))[:, 1]
    etat = em.construire_modele(em.NOMBRES_ETAT).fit(em.preparer(apprentissage, em.NOMBRES_ETAT), apprentissage["perturbe"])
    p_etat = etat.predict_proba(em.preparer(test, em.NOMBRES_ETAT))[:, 1]

    dernier = apprentissage["date"].max()                    # le jour de validation du réseau
    p_res = [entrainer_reseau(apprentissage[apprentissage["date"] < dernier],
                              apprentissage[apprentissage["date"] == dernier], test, g) for g in graines]
    gains_res = [gain(brier(p)) for p in p_res]
    return {
        "nuit": f"{jours_nuit[-1] + pd.Timedelta(days=1):%d/%m}",
        "gain logistique": gain(brier(p_logi)),
        "gain + état ligne": gain(brier(p_etat)),
        "gain réseau (moy.)": np.mean(gains_res),
        "réseau ± écart": np.std(gains_res),
        "réel %": 100 * y.mean(),
        "annoncé logi. %": 100 * p_logi.mean(),
        "annoncé état %": 100 * p_etat.mean(),
        "annoncé réseau %": 100 * np.mean([p.mean() for p in p_res]),
    }

## 7. L'essai : les 10 dernières nuits rejouées
`GRAINES` = nombre d'entraînements du réseau par nuit (départs différents), pour mesurer sa stabilité.
Compter environ **30 à 60 min sur GPU**. Pour un premier essai rapide : `NUITS = 3`, `GRAINES = [0]`.

In [ ]:
NUITS = 10
GRAINES = [0, 1, 2]

resultats = []
for k in range(NUITS, 0, -1):
    jours_nuit = jours[:len(jours) - k + 1]
    if len(jours_nuit) < em.JOURS_TEST + 5:
        continue
    resultats.append(une_nuit(jours_nuit, GRAINES))
    print(resultats[-1]["nuit"], "· réseau", round(resultats[-1]["gain réseau (moy.)"], 2), "%")

tableau = pd.DataFrame(resultats).set_index("nuit").round(2)
tableau

## 8. Le résultat en un coup d'œil

In [ ]:
import matplotlib.pyplot as plt
ax = tableau[["gain logistique", "gain + état ligne", "gain réseau (moy.)"]].plot(marker="o", figsize=(9, 4))
ax.fill_between(range(len(tableau)), tableau["gain réseau (moy.)"] - tableau["réseau ± écart"],
                tableau["gain réseau (moy.)"] + tableau["réseau ± écart"], alpha=0.2)
ax.axhline(3, linestyle="--", color="gray", label="objectif 3 %")
ax.set_ylabel("gain sur le taux de la ligne (%)"); ax.legend(); plt.tight_layout(); plt.show()
print(tableau[["gain logistique", "gain + état ligne", "gain réseau (moy.)"]].agg(["mean", "std"]).round(2))

## 9. Comment lire le résultat
- **Le réseau fait-il mieux que « + état de la ligne », et la plupart des nuits ?** Une seule nuit ne suffit pas.
- **Est-il stable ?** Si l'écart entre graines est grand, il dépend trop du hasard des poids de départ.
- **Annonce-t-il le bon niveau ?** Comparer « annoncé réseau % » à « réel % » (objectif : moins de 2 points d'écart).

Dans tous les cas, c'est une conclusion mesurée pour l'exposé.